# Patch-seq CCF soma positions from registered SWC skeletons

CCF-registered morphologies (`ccf_swc`, `inh_ccf_swc`) were pointed to by Nathan Gouwens and Matt Mallory at `/allen/programs/celltypes/workgroups/ivscc/nathang/repos/slice_morph_to_ccf/ccf_swc`. SWC node coordinates are CCF (µm); the soma is the node with type 1.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../../data/CCF_Patchseq")
FOLDERS = {
    "ccf_swc": "ccf_soma_pos_exc_ccf_swc.csv",
    "inh_ccf_swc": "ccf_soma_pos_inh_ccf_swc.csv",
}

In [2]:
def read_soma(swc_path):
    """Return soma (type 1) node of an SWC as a dict, plus node count."""
    soma, n_nodes = None, 0
    with open(swc_path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            p = line.split()
            n_nodes += 1
            if soma is None and int(p[1]) == 1:
                soma = p
    if soma is None:
        raise ValueError(f"no soma node in {swc_path}")
    return {
        "ccf_soma_x": float(soma[2]),
        "ccf_soma_y": float(soma[3]),
        "ccf_soma_z": float(soma[4]),
        "soma_radius": float(soma[5]),
        "n_nodes": n_nodes,
    }

In [3]:
results = {}
for folder, out_name in FOLDERS.items():
    rows = []
    for swc in sorted((DATA_DIR / folder).glob("*.swc")):
        rows.append({"specimen_id": int(swc.stem), **read_soma(swc)})
    df = pd.DataFrame(rows)
    df.to_csv(DATA_DIR / out_name, index=False)
    results[folder] = df
    print(f"{folder}: {len(df)} cells -> {out_name}")

ccf_swc: 409 cells -> ccf_soma_pos_exc_ccf_swc.csv


inh_ccf_swc: 520 cells -> ccf_soma_pos_inh_ccf_swc.csv


In [4]:
for folder, df in results.items():
    print(folder)
    print(df[["ccf_soma_x", "ccf_soma_y", "ccf_soma_z"]].describe().loc[["min", "max"]].round(0))

ccf_swc
     ccf_soma_x  ccf_soma_y  ccf_soma_z
min      8023.0       417.0      1811.0
max      9973.0      2618.0      9590.0
inh_ccf_swc
     ccf_soma_x  ccf_soma_y  ccf_soma_z
min      7934.0       338.0      1612.0
max      9959.0      2565.0      9889.0
